# Day 2.6 — Evaluating Retrieval Separately from Answers
When an answer is wrong the first question is not "which prompt should I change?" but "did the
right evidence reach the model at all?". A **golden set** turns that into two numbers instead of
one impression.

### Diagnose in this order

1. What was the query? 2. Which chunks came back, with what scores? 3. Is any of them sufficient?
4. Which chunk *should* have appeared? 5. Was good evidence present and unused?
6. Did citation validation accept something we never retrieved?

Steps 1–4 are retrieval, 5–6 the answer layer. Mixing them is how a week disappears into prompt
edits for a chunk that was never retrieved.

In [ ]:
class GoldenCase(BaseModel):
    id: str
    question: str
    answerable: bool
    expected_source: Optional[str]
    expected_section: Optional[str]
    essential_terms: list[str]                    # facts a correct answer must contain

cases = [GoldenCase(**case) for case in GOLDEN_CASES]
for case in cases[:2]:
    print(case.model_dump())
unanswerable = [case for case in cases if not case.answerable]
print("\nanswerable:", len(cases) - len(unanswerable), "| unanswerable:", [case.id for case in unanswerable])
print("expected_source of the unanswerable case:", unanswerable[0].expected_source,
      "-> 'did we retrieve it?' has no answer,")
print("so that check must report n/a. Counting it as a pass would inflate every retrieval rate.")

### Step 1 — Score retrieval alone

No generator is involved: did the expected **source** appear in the top-k, and the expected
**section**? Rates cover only the applicable cases, with the count beside them so `1.0` out of one
case never looks like `1.0` out of ten.

In [ ]:
def evaluate_retrieval(index, cases, top_k=3):
    """Did the expected evidence reach the top-k list? Generation is not involved."""
    records = []
    for case in cases:
        hits = index.search(case.question, top_k=top_k)
        if case.answerable:
            source_hit = case.expected_source in [item.chunk.source for item in hits]
            match = [item for item in hits
                     if item.chunk.source == case.expected_source and item.chunk.section == case.expected_section]
            section_hit, expected_rank = bool(match), (match[0].rank if match else None)
        else:
            source_hit = section_hit = expected_rank = None      # nothing expected: a rate would be fiction
        records.append({"id": case.id, "answerable": case.answerable, "source_hit": source_hit,
                        "section_hit": section_hit, "expected_rank": expected_rank,
                        "retrieved_ids": [item.chunk.chunk_id for item in hits],
                        "retrieved_sections": [item.chunk.section for item in hits]})
    return records

def summarize(records, fields):
    """Rate per field over the records where it applies; None is skipped, never counted."""
    summary = {}
    for field in fields:
        applies = [r for r in records if r.get(field) is not None]
        summary[field] = round(sum(bool(r[field]) for r in applies) / len(applies), 3) if applies else 0.0
    return summary

def summarize_detail(records, fields):
    """The same numbers as 'hits/applicable (n n/a)' so a rate can never mislead."""
    detail = {}
    for field in fields:
        applies = [r for r in records if r.get(field) is not None]
        skipped = len(records) - len(applies)
        detail[field] = f"{sum(bool(r[field]) for r in applies)}/{len(applies)}" + (f" ({skipped} n/a)" if skipped else "")
    return detail

def render_table(records, columns):
    """A small aligned text table: None prints as n/a, booleans as yes/NO."""
    def cell(value):
        if value is None:
            return "n/a"
        if isinstance(value, bool):
            return "yes" if value else "NO"
        if isinstance(value, float):
            return f"{value:.2f}"
        return ", ".join(map(str, value)) if isinstance(value, list) else str(value)
    widths = [max(len(column), *(len(cell(r.get(column))) for r in records)) for column in columns]
    lines = ["  ".join(c.ljust(w) for c, w in zip(columns, widths)),
             "  ".join("-" * w for w in widths)]
    lines += ["  ".join(cell(r.get(c)).ljust(w) for c, w in zip(columns, widths)) for r in records]
    return "\n".join(lines)

retrieval_hash = evaluate_retrieval(hash_index, cases, top_k=3)
print(render_table(retrieval_hash, ["id", "answerable", "source_hit", "section_hit", "expected_rank"]))
print("\nrates :", summarize(retrieval_hash, ["source_hit", "section_hit"]))
print("counts:", summarize_detail(retrieval_hash, ["source_hit", "section_hit"]))

### Step 2 — Look at the misses, then change one layer

A rate is a pointer, not a diagnosis. Print the cases whose expected section never reached the
top-3; they are a *representation* problem, so swap the embedder and re-run the identical set.

In [ ]:
by_id = {case.id: case for case in cases}
misses = [record for record in retrieval_hash if record["answerable"] and not record["section_hit"]]
print("cases missing their expected section:", [record["id"] for record in misses])
for record in misses:
    case = by_id[record["id"]]
    print(f"\n{case.id}: {case.question}")
    print("   expected :", case.expected_source, "|", case.expected_section)
    for chunk_id, section in zip(record["retrieved_ids"], record["retrieved_sections"]):
        print("   retrieved:", chunk_id, "|", section)

print("\nSAME SET, EMBEDDER SWAPPED")
if semantic_index is None:
    print("Semantic embedder unavailable here, so this comparison cannot run. With")
    print("sentence-transformers installed, the misses above are expected to disappear.")
else:
    retrieval_semantic = evaluate_retrieval(semantic_index, cases, top_k=3)
    print("hash     :", summarize(retrieval_hash, ["source_hit", "section_hit"]),
          summarize_detail(retrieval_hash, ["section_hit"]))
    print("semantic :", summarize(retrieval_semantic, ["source_hit", "section_hit"]),
          summarize_detail(retrieval_semantic, ["section_hit"]))
    for old, new in zip(retrieval_hash, retrieval_semantic):
        if old["section_hit"] != new["section_hit"]:
            print(f"   {old['id']}: section_hit {old['section_hit']} -> {new['section_hit']}"
                  f"  (expected chunk rank {old['expected_rank']} -> {new['expected_rank']})")

### Step 3 — Does a bigger top-k fix everything?

Raising top-k can only help recall, but every extra chunk costs tokens and adds a distractor the
generator may quote instead.

In [ ]:
def section_hit_rate(index, k):
    records = [r for r in evaluate_retrieval(index, cases, top_k=k) if r["answerable"]]
    return sum(bool(r["section_hit"]) for r in records) / len(records)

average_chars = sum(len(chunk.text) for chunk in chunks) / len(chunks)
print(f"{'top_k':8}{'hash':10}{'semantic':12}{'context sent':>14}")
for k in (1, 2, 3, 5):
    semantic_value = f"{section_hit_rate(semantic_index, k):.2f}" if semantic_index else "n/a"
    print(f"{k:<8}{section_hit_rate(hash_index, k):<10.2f}{semantic_value:<12}{int(k * average_chars):>9} chars")
print("\nRecall stops improving long before the cost does. top_k is a budget decision, not a quality dial.")

### Step 4 — Now score the answers, separately

Different questions entirely: did it abstain when it should, cite the expected source, keep every
citation through validation, and state the facts? That last one, **essential-term coverage**, stops
a correct citation with an empty answer from scoring as a pass.

In [ ]:
def evaluate_answers(assistant, cases):
    """Did the answer abstain correctly, cite retrieved evidence, and state the essential facts?"""
    records = []
    for case in cases:
        state = assistant.answer(case.question)
        answer = state.answer
        text = answer.answer.lower() if answer else ""
        found = [term for term in case.essential_terms if term.lower() in text]
        cited = [citation.source for citation in answer.citations] if answer else []
        records.append({
            "id": case.id, "answerable": case.answerable, "completed": state.status == "completed",
            "abstained": bool(answer) and answer.abstained,
            "abstention_correct": bool(answer) and answer.abstained == (not case.answerable),
            "citation_correct": bool(answer) and ((not case.answerable and not answer.citations)
                                                  or (case.answerable and case.expected_source in cited)),
            # provenance is OUR check: every citation survived validation against the retrieval log
            "citation_provenance_ok": bool(answer) and answer.grounded,
            "dropped_citations": len(answer.dropped_citations) if answer else 0,
            "essential_terms_found": len(found), "essential_terms_total": len(case.essential_terms),
            "essential_term_coverage": (len(found) / len(case.essential_terms)) if case.essential_terms else None,
            "missing_terms": [term for term in case.essential_terms if term not in found],
            "error": state.error})
    return records

def summarize_essential_terms(records):
    scored = [r for r in records if r["essential_terms_total"]]
    found, total = sum(r["essential_terms_found"] for r in scored), sum(r["essential_terms_total"] for r in scored)
    return {"cases_scored": len(scored), "terms_found": found, "terms_total": total,
            "coverage": round(found / total, 3) if total else 0.0}

ANSWER_FIELDS = ["completed", "abstention_correct", "citation_correct", "citation_provenance_ok"]
answers_hash = evaluate_answers(KnowledgeAssistant(hash_index, top_k=3), cases)
print(render_table(answers_hash, ["id", "answerable", "abstained", "abstention_correct",
                                  "citation_correct", "citation_provenance_ok", "essential_term_coverage"]))
print("\nrates :", summarize(answers_hash, ANSWER_FIELDS))
print("counts:", summarize_detail(answers_hash, ANSWER_FIELDS))
print("terms :", summarize_essential_terms(answers_hash))

print("\nWHERE THE TWO REPORTS DISAGREE, AND WHICH LAYER OWNS IT")
section_hits = {r["id"]: r["section_hit"] for r in retrieval_hash}
for record in answers_hash:
    if record["answerable"] and record["abstained"]:
        print(record["id"], "abstained although the corpus contains the answer -> retrieval never supplied")
        print("     the section, so abstaining was the safest thing it could do. Layer: RETRIEVAL.")
    elif record["essential_terms_total"] and record["essential_term_coverage"] == 0:
        arrived = section_hits[record["id"]]
        print(record["id"], "states none of its essential terms:", record["missing_terms"])
        print("     section_hit was", "yes -> Layer: GENERATION (the evidence arrived and was not used)."
              if arrived else "NO -> Layer: RETRIEVAL (right file, wrong section).")

### Try it yourself

Predict whether `top_k=5` repairs the hash embedder's missing sections, and what it costs. Check
both scorecards, not only the one you hoped would move.

In [ ]:
# --- Worked solution ---------------------------------------------------------------
wide = evaluate_retrieval(hash_index, cases, top_k=5)
for old, new in zip(retrieval_hash, wide):
    if old["section_hit"] != new["section_hit"]:
        print(f"retrieval {old['id']}: section_hit {old['section_hit']} -> {new['section_hit']}"
              f" (rank {old['expected_rank']} -> {new['expected_rank']})")

answers_wide = evaluate_answers(KnowledgeAssistant(hash_index, top_k=5), cases)
print("\nanswers at top_k=3:", summarize(answers_hash, ANSWER_FIELDS))
print("answers at top_k=5:", summarize(answers_wide, ANSWER_FIELDS))
print("terms   at top_k=3:", summarize_essential_terms(answers_hash))
print("terms   at top_k=5:", summarize_essential_terms(answers_wide))
for old, new in zip(answers_hash, answers_wide):
    if old["essential_term_coverage"] != new["essential_term_coverage"]:
        print(f"   {old['id']}: term coverage {old['essential_term_coverage']} -> {new['essential_term_coverage']}")
print("\nMore context can recover a missing section AND hand the generator more chances to quote the")
print("wrong one. Always re-measure both halves after a change.")

### Checkpoint

**1. Why is the unanswerable case reported as `n/a` instead of a hit?**

<details><summary>Show answer</summary>

It has no expected source or section, so *did we retrieve it?* has no answer. Counting it as a pass would raise every retrieval rate by ten percent and hide a real miss.

</details>

**2. Retrieval scored 7/9 but the answers scored higher on citations. Which number should you act on?**

<details><summary>Show answer</summary>

The retrieval number. A citation can be right at file level while the quoted section is wrong, and fixing generation cannot recover a section that was never retrieved.

</details>

### Recap

- **Limitation seen:** one average hides which layer failed, and an inapplicable case counted as a pass inflates it.
- **Layer added:** separate retrieval and answer reports, `n/a` where a check cannot apply, essential-term coverage, a top-k sweep.
- **Evidence:** the misses are named by id, swapping only the embedder repairs them, and the answer table moves with them.